# Appendix C — Notation, PCA and references

## C.1 Notation used throughout

| symbol | meaning | unit |
|---|---|---|
| $S_t$ | price of the exposure we hold | EUR/MWh |
| $F_t$, $\mathbf{F}_t$ | price(s) of hedge instrument(s) | EUR/MWh |
| $\Delta X_t = X_t - X_{t-1}$ | one-day change | EUR/MWh |
| $Q$, $Q_S$ | exposure size (positive = long) | MWh |
| $H_{y,m}$ | delivery hours of month $(y,m)$; MWh per lot | h |
| $n_{\text{lots}}$ | number of futures lots | — |
| $h$, $h^*$ | hedge ratio; minimum-variance ratio | MWh of $F$ per MWh of $S$ |
| $\mathbf{w}$ | vector of hedge sizes across instruments | MWh |
| $\mathbf{q}$, $\boldsymbol\Delta$ | delta ladder (MWh per tenor) | MWh |
| $\Pi$, $\Delta V$ | portfolio P&L | EUR |
| $\sigma_X$, $\rho$ | standard deviation of $\Delta X$; correlation | EUR/MWh; — |
| $\Sigma$ | covariance matrix of $\Delta\mathbf{P}$ | (EUR/MWh)² |
| $\Sigma_{FF}$, $\Sigma_{FS}$ | hedge-instrument covariance; cross-covariance with exposure | (EUR/MWh)² |
| $b_t = S_t - hF_t$ | (generalised) basis | EUR/MWh |
| $\kappa$, $\mu$, $t_{1/2}$ | OU mean-reversion speed, long-run mean, half-life | 1/day; EUR/MWh; days |
| $\lambda$ | risk aversion (mean–variance) | 1/EUR |
| $c$ | proportional transaction cost | EUR/MWh or EUR/lot |
| $B$ | hedging band | lots |
| $v_k$, $\lambda_k$ | PCA loading vector and eigenvalue of component $k$ | —; (EUR/MWh)² |
| $\beta_k = \mathbf{q}^\top v_k$ | factor exposure | MWh (loading-weighted) |
| $\text{VaR}_\alpha$, $\text{ES}_\alpha$ | value at risk / expected shortfall at confidence $\alpha$ | EUR |
| $z_\alpha$, $\varphi$ | standard normal quantile and density | — |

Conventions: *long* exposure means we own gas (gain when prices rise). A hedge ratio $h$ means we are
**short** $hQ$ MWh of $F$ against a long $Q$ MWh of $S$. All statistics are on **daily changes** in
EUR/MWh unless stated otherwise.

## C.2 Principal component analysis, in enough detail to implement

Given $n$ daily change vectors $\Delta\mathbf{P}_t \in \mathbb{R}^k$, form the sample covariance
$\Sigma$ ($k\times k$, symmetric positive semi-definite). The spectral theorem gives

$$
\Sigma = V\Lambda V^\top, \qquad V^\top V = I, \qquad \Lambda = \text{diag}(\lambda_1 \ge \dots \ge \lambda_k \ge 0).
\tag{C.1}
$$

Columns of $V$ are the **loadings** (eigenvectors); $\lambda_j$ is the variance of the $j$-th
**score** $z_{j,t} = v_j^\top\Delta\mathbf{P}_t$. Scores are uncorrelated:
$\text{Cov}(z_i, z_j) = v_i^\top\Sigma v_j = \lambda_j\,v_i^\top v_j = \lambda_j\delta_{ij}$.

**Variance explained** by the first $m$ components: $\sum_{j\le m}\lambda_j / \sum_j \lambda_j$.

**Reconstruction**: $\Delta\mathbf{P}_t = \sum_j v_j z_{j,t}$; truncating at $m$ gives the best rank-$m$
approximation in mean-squared error (Eckart–Young).

**Portfolio variance in factor coordinates**: for a delta vector $\mathbf{q}$,
$\mathbf{q}^\top\Sigma\mathbf{q} = \mathbf{q}^\top V\Lambda V^\top\mathbf{q} = \sum_j \lambda_j(\mathbf{q}^\top v_j)^2 = \sum_j\lambda_j\beta_j^2$ — equation (3.15).

**Sign convention**: eigenvectors are defined up to sign. `gashedge.risk.pca` flips each so that its
elements sum to a positive number, making PC1 a "prices up" shock.

**Covariance vs correlation PCA**: on a covariance matrix the front tenors (high vol) dominate PC1; on a
correlation matrix all tenors are weighted equally. For *hedging in EUR* use covariance; for *describing
shape* use correlation.

**Why level/slope/curvature appear**: for any smooth, positive, decaying correlation structure along a
line (tenors), the leading eigenvectors of the covariance are approximately the first few orthogonal
polynomials in tenor — constant, linear, quadratic. This is a property of the mathematics, not of gas;
it appears in interest-rate curves, oil curves and volatility surfaces alike.

## C.3 Numerical illustration

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "gashedge").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from datetime import date
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gashedge import get_logger
from gashedge.plotting import setup_style

from gashedge.market_data import simulate_curve_history
from gashedge.risk import pca
setup_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
log = get_logger("appendixC")
log.info("Environment ready")

2026-09-13 16:47:23.885 | INFO    | appendixC | Environment ready


In [2]:
d = simulate_curve_history(n_days=750, n_tenors=24).diff().dropna()
res = pca(d, n_components=3)
V, lam = res["loadings"].values, res["eigval"]
Sigma = d.cov().values
recon = (res["loadings"] * lam[:3]) @ res["loadings"].T          # rank-3 reconstruction of Sigma
err = np.linalg.norm(Sigma - recon.values) / np.linalg.norm(Sigma)
q = np.zeros(24); q[0], q[11] = 10_000, -20_000
log.info("Rank-3 Sigma reconstruction error %.1f%% (Frobenius); q'Sigma q = %.0f vs sum lambda_j beta_j^2 (all 24) = %.0f",
         100 * err, q @ Sigma @ q, sum(l * (q @ v) ** 2 for l, v in zip(res["eigval"], np.linalg.eigh(Sigma)[1][:, ::-1].T)))
pd.DataFrame({"eigenvalue": lam[:5], "explained": lam[:5] / lam.sum(), "cumulative": np.cumsum(lam[:5]) / lam.sum()},
             index=[f"PC{i + 1}" for i in range(5)]).round(4)

2026-09-13 16:47:23.893 | INFO    | gashedge.market_data | Simulated constant-maturity curve history: 750 days x 24 tenors


2026-09-13 16:47:23.894 | INFO    | gashedge.risk | PCA explained variance: [0.819 0.115 0.013]


2026-09-13 16:47:23.895 | INFO    | appendixC | Rank-3 Sigma reconstruction error 1.7% (Frobenius); q'Sigma q = 85277364 vs sum lambda_j beta_j^2 (all 24) = 85277364


,eigenvalue,explained,cumulative
PC1,10.2996,0.8188,0.8188
PC2,1.4527,0.1155,0.9342
PC3,0.1573,0.0125,0.9467
PC4,0.0964,0.0077,0.9544
PC5,0.0644,0.0051,0.9595


## C.4 References and further reading

**Contracts and market structure**
* ICE Endex, *Dutch TTF Natural Gas Futures — contract specifications* (exchange website; check lot size,
  delivery hours, last trading day and holiday calendar before production use).
* ACER / CEER, *European Gas Market Monitoring Reports* — hub liquidity and churn statistics.
* Heather, P. (2020), *European Traded Gas Hubs: the supremacy of TTF*, Oxford Institute for Energy Studies.

**Hedging theory**
* Ederington, L. (1979), "The Hedging Performance of the New Futures Markets", *Journal of Finance* —
  the minimum-variance hedge ratio and effectiveness as $R^2$.
* Hull, J., *Options, Futures and Other Derivatives*, ch. 3 (hedging with futures; basis; stack-and-roll).
* Geman, H. (2005), *Commodities and Commodity Derivatives* — seasonality, storage, gas and power.
* Samuelson, P. (1965), "Proof that Properly Anticipated Prices Fluctuate Randomly", *Industrial Management
  Review* — origin of the volatility term-structure effect.

**Metallgesellschaft**
* Culp, C. & Miller, M. (1994), "Hedging a Flow of Commodity Deliveries with Futures", *Derivatives Quarterly*.
* Mello, A. & Parsons, J. (1995), "Maturity Structure of a Hedge Matters: Lessons from the Metallgesellschaft
  Debacle", *Journal of Applied Corporate Finance*.
* Edwards, F. & Canter, M. (1995), "The Collapse of Metallgesellschaft", *Journal of Futures Markets*.

**Execution and inventory**
* Almgren, R., Thum, C., Hauptmann, E. & Li, H. (2005), "Direct Estimation of Equity Market Impact", *Risk*.
* Tóth, B. et al. (2011), "Anomalous Price Impact and the Critical Nature of Liquidity in Financial Markets",
  *Physical Review X* — the square-root law.
* Avellaneda, M. & Stoikov, S. (2008), "High-frequency Trading in a Limit Order Book", *Quantitative Finance*.
* Whalley, A. E. & Wilmott, P. (1997), "An Asymptotic Analysis of an Optimal Hedging Model for Option
  Pricing with Transaction Costs", *Mathematical Finance* — cube-root hedging bands.

**Risk measurement**
* Jorion, P., *Value at Risk*, 3rd ed. — parametric/historical VaR, backtesting.
* Kupiec, P. (1995), "Techniques for Verifying the Accuracy of Risk Measurement Models", *Journal of Derivatives*.
* Christoffersen, P. (1998), "Evaluating Interval Forecasts", *International Economic Review*.
* Litterman, R. & Scheinkman, J. (1991), "Common Factors Affecting Bond Returns", *Journal of Fixed Income* —
  level/slope/curvature PCA.
* Ledoit, O. & Wolf, M. (2004), "Honey, I Shrunk the Sample Covariance Matrix", *Journal of Portfolio
  Management* — shrinkage for $\Sigma$.

**Econometrics**
* Granger, C. & Newbold, P. (1974), "Spurious Regressions in Econometrics", *Journal of Econometrics*.
* Hamilton, J., *Time Series Analysis* — AR(1)/OU estimation, cointegration.

---
*End of the tutorial. Back to the [Introduction](../00_introduction.ipynb).*

---
◀ [Previous](B_delivery_hours_and_strips.ipynb) · [Contents](../00_introduction.ipynb)